# Electricity Consumption Analysis and Demand Prediction

Companion academic notebook for data profiling, feature engineering, time-based EDA, Spark aggregation, and regression evaluation. Start with the included sample CSV or set `CSV_PATH` to a real electricity dataset. The dashboard and API perform the same workflow interactively.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CSV_PATH = Path('../backend/data/sample_electricity.csv')
if not CSV_PATH.exists():
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(27)
    dates = pd.date_range('2024-01-01', periods=24 * 366, freq='h')
    hour = dates.hour.to_numpy()
    season = np.sin(2 * np.pi * (dates.dayofyear.to_numpy() - 105) / 366)
    temperature = 17 + 10 * season + 4 * np.sin(2 * np.pi * (hour - 8) / 24)
    demand = (34 + 18 * np.exp(-0.5 * ((hour - 19) / 3) ** 2) + 0.8 * np.maximum(temperature - 24, 0) + rng.normal(0, 3, len(dates))).clip(5)
    pd.DataFrame({'timestamp': dates, 'consumption_kwh': demand, 'temperature_c': temperature}).to_csv(CSV_PATH, index=False)

raw = pd.read_csv(CSV_PATH)
raw.head()

## Profile and prepare

The notebook infers common timestamp and consumption names, removes duplicates, fills numeric gaps, applies IQR capping to the target, and derives calendar features. Update `TIME_COLUMN` or `TARGET_COLUMN` for unusual schemas.

In [ ]:
TIME_COLUMN = next((c for c in raw.columns if any(s in c.lower() for s in ('timestamp', 'datetime', 'date', 'time'))), raw.columns[0])
TARGET_COLUMN = next((c for c in raw.columns if any(s in c.lower() for s in ('consumption', 'demand', 'load', 'energy', 'power', 'kwh'))), None)
if TARGET_COLUMN is None:
    candidates = raw.select_dtypes(include='number').columns.tolist()
    if not candidates:
        raise ValueError('No numeric consumption column found; set TARGET_COLUMN explicitly.')
    TARGET_COLUMN = max(candidates, key=lambda c: raw[c].var())

df = raw.drop_duplicates().copy()
df['_timestamp'] = pd.to_datetime(df[TIME_COLUMN], errors='coerce', format='mixed')
df['_target'] = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')
df = df.dropna(subset=['_timestamp']).sort_values('_timestamp')
df['_target'] = df['_target'].interpolate(limit_direction='both')
q1, q3 = df['_target'].quantile([.25, .75])
iqr = q3 - q1
if iqr > 0:
    df['_target'] = df['_target'].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
df['hour'] = df['_timestamp'].dt.hour
df['day'] = df['_timestamp'].dt.day
df['day_of_week'] = df['_timestamp'].dt.dayofweek
df['week'] = df['_timestamp'].dt.isocalendar().week.astype(int)
df['month'] = df['_timestamp'].dt.month
df['year'] = df['_timestamp'].dt.year
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['season'] = df['_timestamp'].dt.month.map({12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Autumn',10:'Autumn',11:'Autumn'})
print({'rows': len(df), 'columns': list(raw.columns), 'target': TARGET_COLUMN, 'missing_before_cleaning': raw.isna().sum().to_dict()})
df['_target'].describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
df.set_index('_timestamp')['_target'].resample('D').mean().plot(ax=axes[0, 0], color='#16735b', title='Daily mean consumption')
df.groupby('hour')['_target'].mean().plot(ax=axes[0, 1], color='#da8d36', title='Average demand by hour')
df.groupby('day_of_week')['_target'].mean().plot(kind='bar', ax=axes[1, 0], color='#4387a5', title='Average demand by weekday')
sns.histplot(df['_target'], bins=24, ax=axes[1, 1], color='#8457a6')
axes[1, 1].set_title('Consumption distribution')
plt.tight_layout()
plt.show()

## PySpark Big Data demonstration

Spark reads the CSV into a distributed DataFrame, cleans missing/duplicate timestamp and target values, derives time fields, and executes grouped aggregations. This is the scalable analytics stage; only a bounded sample should be converted to Pandas for the scikit-learn model.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master('local[2]').appName('ElectricityDemandNotebook')
         .config('spark.ui.enabled', 'false').config('spark.sql.shuffle.partitions', '4').getOrCreate())
spark_df = spark.read.option('header', True).option('inferSchema', True).csv(str(CSV_PATH.resolve()))
spark_clean = (spark_df.withColumn('_timestamp', F.to_timestamp(F.col(TIME_COLUMN)))
               .withColumn('_demand', F.col(TARGET_COLUMN).cast('double'))
               .dropna(subset=['_timestamp', '_demand']).dropDuplicates()
               .withColumn('hour', F.hour('_timestamp'))
               .withColumn('day_of_week', F.dayofweek('_timestamp'))
               .withColumn('month', F.month('_timestamp'))
               .withColumn('year', F.year('_timestamp')))
spark_daily = spark_clean.groupBy(F.to_date('_timestamp').alias('day')).agg(F.sum('_demand').alias('daily_demand'))
spark_monthly = spark_clean.groupBy(F.date_format('_timestamp', 'yyyy-MM').alias('month')).agg(F.sum('_demand').alias('monthly_demand'))
print({'rows_after_spark_cleaning': spark_clean.count(), 'daily_groups': spark_daily.count(), 'monthly_groups': spark_monthly.count()})
spark_daily.orderBy('day').show(10, truncate=False)
spark.stop()

## Time-ordered model comparison

Lagged consumption and calendar/weather features are used when available. The last 20% remains held out to avoid random shuffling across time.

In [ ]:
for lag in (1, 24, 168):
    df[f'lag_{lag}'] = df['_target'].shift(lag)
df[['lag_1', 'lag_24', 'lag_168']] = df[['lag_1', 'lag_24', 'lag_168']].bfill().fillna(df['_target'].median())
features = ['hour', 'day', 'day_of_week', 'week', 'month', 'year', 'is_weekend', 'lag_1', 'lag_24', 'lag_168']
weather = next((c for c in df.select_dtypes(include='number').columns if any(s in c.lower() for s in ('temp', 'weather', 'humidity', 'pressure'))), None)
if weather:
    features.append(weather)
split = int(len(df) * .8)
x_train, x_test = df[features].iloc[:split], df[features].iloc[split:]
y_train, y_test = df['_target'].iloc[:split], df['_target'].iloc[split:]
models = {
    'Linear Regression': make_pipeline(StandardScaler(), LinearRegression()),
    'Random Forest': RandomForestRegressor(n_estimators=80, max_depth=18, min_samples_leaf=2, random_state=31, n_jobs=2),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=90, max_depth=2, learning_rate=.06, loss='huber', random_state=31),
}
results = []
for name, estimator in models.items():
    estimator.fit(x_train, y_train)
    predicted = estimator.predict(x_test)
    mse = mean_squared_error(y_test, predicted)
    results.append({'model': name, 'MAE': mean_absolute_error(y_test, predicted), 'MSE': mse,
                    'RMSE': np.sqrt(mse), 'R2': r2_score(y_test, predicted)})
performance = pd.DataFrame(results).sort_values('RMSE')
performance